# Use this notebook to visualise subsets / instantaneous data

**How to use:**
1. run the 3 cells in import and csv path. This will import the necessary packages, and find the LED file as well as the folder of csvs to be analysed. Here, you can check if the correct number of csv files are being detected.
Typically, changing the experiment name should work if you have already worked with it before. However, when working with a new experiment or using an updated csv folder/LED times csv, we need to update the paths one time in the EXPERIMENT_SET cell.

2. Next, we create the subset for the LED duration. The dataframe can be changed according to the requirements. After that, we create a similar dataframe for all the files for a random LED duration but before the LED to establish an animal baseline. the random LED duration is not needed if we are not comparing the animal against itself.
3. groupby is used extensively for plotting. This allows us to separate the different categories (day 1 uro, day 1 control ...) and then plot separate graphs.

For each plot, a **new df** is created which is a subset of the original df. This is done to ensure we do not edit the original df. After this, groupby is applied and its plotted.

The visualisations are saved to ./data/visualisation/<EXPERIMENT_SET>/<type_of_plot>/ csv / svg / jpg


## Imports + csv paths

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import random
import os
from scripts.visualisation import *
plt.style.use('seaborn')


Change the paths in the two cells below to point to the correct LED times file, and the correct output csvs (these should be after main.ipynb)



In [ ]:
EXPERIMENT_SET ="d48"

led_csv_path =Path("./data/output/d48_LED_times.csv")
dlc_csv_analysed_path = Path("./data/output/d48_vr03_final/")

print(EXPERIMENT_SET)

In [ ]:
led_csv = pd.read_csv(led_csv_path)
if led_csv.empty:
    print("LED csv is empty")
else:
    # number of unique rows in the csv column 1
    print(f"LED data found for {led_csv['name'].nunique()} csv files")

In [ ]:
csv_files = list(dlc_csv_analysed_path.glob("**/*.csv"))
print(len(csv_files), "csv files found")

In [ ]:
# print if csv files and led csv files are the same, print the ones not present in either
csv_files_names = [os.path.basename(csv_file).rstrip('.csv') for csv_file in csv_files]
led_csv_names = [i.rstrip('.mp4') for i in led_csv['name'].unique().tolist()]
print("csv files not in led csv" , set(csv_files_names) - set(led_csv_names))
print("led times not in csv files" , set(led_csv_names) - set(csv_files_names))


## Setup Dataframes
we set up the LED df, as well as the random 3s before LED dataframe below. 

The metrics recorded are cumulative distance travelled, avg vel, avg acc, avg head-tail angle, minimum distance to LED (max height reached), % of frames above initial, magnitude of y travelled (from start to highest point), start to highest trajectory angle, 

The create_master_dataframe checks which columns are needed, and creates a master dataframe that is used downstream for visualisations. This is done so that we do not have to go back and create the dataframe for every new visualisation of the same experiment set data.

For creating new metrics, they have to be declared in create_master_dataframe and columns


In [ ]:
# These are the data points created for analysis 
columns = ['name',
 'uroa_control',
 'day',
 'dist',
 'vel',
 'acc',
 'head_angle',
 'dist_surface',
 'y_up',
 'y_mag',
 'trajectory_angle',
 'start_y_from_led',
 'end_trajectory_angle']


In [ ]:
# parameters with their meanings 
parameter_dict = {
    'dist': 'total distance travelled',
    'vel': 'average velocity',
    'acc': 'average acceleration',
    'head_angle': 'average tail-head angle',
    'dist_surface': 'distance from highest point in trajectory to LED',
    'y_up' : 'percentage of frames with y above starting y',
    'y_mag': 'magnitude of y between start and highest point',
    'trajectory_angle': 'angle between start and highest point',
    'start_y_from_led': 'distance from LED to start point',
    'end_trajectory_angle': 'angle between start and end point'
}


#### LED subset data frame 

In [ ]:
led_df = create_master_dataframe(columns, csv_files, led_csv, random_pre_led = False)

#### creating random subset before LED

In [ ]:
before_led_df = create_master_dataframe(columns, csv_files, led_csv, random_pre_led = True, debug = False)

## Normal plots (sns)

In [ ]:
save_path = Path(f"./data/visualisation/{EXPERIMENT_SET}/")

### LED Distance/Vel and acceleration box plot

In [ ]:
# create a dataframe with the data points for the led data
dist_vel_acc_df = led_df[["name","day", "uroa_control", "dist", "vel", "acc"]]

#save the dataframe
dist_vel_acc_save_path = save_path/Path(f"LED_dist_vel_acc_hist")
os.makedirs(dist_vel_acc_save_path, exist_ok = True)
dist_vel_acc_df.to_csv(dist_vel_acc_save_path/"dist_vel_acc_df.csv", index = False)

# group the data by uroa control and day
dist_vel_acc_df = dist_vel_acc_df.groupby(["uroa_control", "day"])
keys = list(dist_vel_acc_df.groups.keys())

# remove the day 5 data for uroa control if present (was there in week_12)
keys.remove((0, 5)) if (0, 5) in keys else None
keys.remove((1, 5)) if (1, 5) in keys else None

for key in keys:
    df = dist_vel_acc_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")

In [ ]:
# for distance plots, create a 2x4 grid of plots
fig, axes = plt.subplots(2, 4, sharex=True, sharey = True, figsize=(16,8))
plt.suptitle("Distance travelled, control vs uroa", fontsize=16) 
for i, axes in enumerate(axes.flat):
    key = keys[i]
    day_df = dist_vel_acc_df.get_group(key)
    bins = np.linspace(0, 1500, 20)
    # print(day_df.head(5))
    sns.histplot(day_df["dist"],  ax=axes, bins = bins)
    axes.set_title(f"Day {key[1]}")
    axes.set_xlabel("total distance travelled")
    axes.set_ylabel("number of fish")

#save png and svg
plt.savefig(dist_vel_acc_save_path/"dist_hist.png")
plt.savefig(dist_vel_acc_save_path/"dist_hist.svg")

In [ ]:
# for vel plots
fig, axes = plt.subplots(2, 4, sharex=True, sharey = True, figsize=(16,8))
plt.suptitle("avg velocity, control vs uroa", fontsize=16) 
for i, axes in enumerate(axes.flat):
    key = keys[i]
    day_df = dist_vel_acc_df.get_group(key)
    bins = np.linspace(0, 20, 20)
    # print(day_df.head(5))
    sns.histplot(day_df["vel"], ax=axes, bins = bins)
    axes.set_title(f"Day {key[1]}")
    axes.set_xlabel("avg velocity")
    axes.set_ylabel("number of fish")


#save png and svg
plt.savefig(dist_vel_acc_save_path/"avg_vel_hist.png")
plt.savefig(dist_vel_acc_save_path/"avg_vel_hist.svg")

In [ ]:
# for acc plots
fig, axes = plt.subplots(2, 4, sharex=True, sharey = True, figsize=(16,8))
plt.suptitle("avg acc, control vs uroa", fontsize=16) 
for i, axes in enumerate(axes.flat):
    key = keys[i]
    day_df = dist_vel_acc_df.get_group(key)
    bins = np.linspace(0, 500, 20)
    # print(day_df.head(5))
    sns.histplot(day_df["acc"], ax=axes, bins = bins)
    axes.set_title(f"Day {key[1]}")
    axes.set_xlabel("avg acc")
    axes.set_ylabel("number of fish")

#save png and svg
plt.savefig(dist_vel_acc_save_path/"avg_acc_hist.png")
plt.savefig(dist_vel_acc_save_path/"avg_acc_hist.svg")

### Polar plot for head direction in LED

In [ ]:
#create new df with only the angle data
angle_df = led_df[["name","day", "uroa_control", "head_angle"]]

#save df
head_direction_save_path = save_path/Path("LED_head_direction")
os.makedirs(head_direction_save_path, exist_ok = True)
angle_df.to_csv(head_direction_save_path/"angle_df.csv", index = False)

#group by day and uroa_control
angle_df = angle_df.groupby(["uroa_control", "day"])

# remove the keys for day 5
keys = list(angle_df.groups.keys())
keys.remove((0,5)) if (0,5) in keys else None
keys.remove((1,5)) if (1,5) in keys else None

for key in keys:
    df = angle_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")


In [ ]:
#create a separate polar plot for each day
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), sharex=True, sharey=True, figsize=(16,10))
plt.suptitle("average head direction, control vs uroa for each day", fontsize=16)

for i, ax in enumerate(axs.flat):
    group = angle_df.get_group(keys[i])
    angles = group["head_angle"].apply(lambda x: x * (np.pi) / 180)  # convert to radians
    # angles = group["angle"]
    bins = np.linspace(0, 2*np.pi, num=20)
    # bins = np.linspace(0, 360, num=20)
    ax.hist(angles, bins=bins, edgecolor="black", linewidth=0.75)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}")

#save png and svg
plt.savefig(head_direction_save_path/"head_direction.png")
plt.savefig(head_direction_save_path/"head_direction.svg")

### polar plot of trajectory angle

In [ ]:
#create new df with only the angle data
t_angle_df = led_df[["name", "day", "uroa_control", "trajectory_angle"]]

#save df
trajectory_direction_save_path = save_path/Path("trajectory_direction")
os.makedirs(trajectory_direction_save_path, exist_ok = True)
t_angle_df.to_csv(trajectory_direction_save_path/"trajectory_angle_df.csv", index = False)

#group by day and uroa_control
t_angle_df = t_angle_df.groupby(["uroa_control", "day"])

# remove the keys for day 5
keys = list(t_angle_df.groups.keys())
keys.remove((0,5)) if (0,5) in keys else None
keys.remove((1,5)) if (1,5) in keys else None

for key in keys:
    df = t_angle_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")


In [ ]:
#create a separate polar plot for each day
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), sharex=True, sharey=True, figsize=(16,10))
plt.suptitle("trajectory direction, control vs uroa for each day", fontsize=16)

for i, ax in enumerate(axs.flat):
    group = t_angle_df.get_group(keys[i])
    angles = group["trajectory_angle"].apply(lambda x: x * (np.pi) / 180)  # convert to radians
    # angles = group["angle"]
    bins = np.linspace(0, 2*np.pi, num=20)
    # bins = np.linspace(0, 360, num=20)
    ax.hist(angles, bins=bins, edgecolor="black", linewidth=0.75)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}")

#save png and svg
plt.savefig(trajectory_direction_save_path/"trajectory_direction.png")
plt.savefig(trajectory_direction_save_path/"trajectory_direction.svg")

### polar plot for trajectory till end of led 

In [ ]:
#create new df with only the angle data
t_end_angle_df = led_df[["name", "day", "uroa_control", "end_trajectory_angle"]]

#save df
end_trajectory_save_path = save_path/Path("end_trajectory_direction")
os.makedirs(end_trajectory_save_path, exist_ok = True)
t_end_angle_df.to_csv(end_trajectory_save_path/"angle_df.csv", index = False)

#group by day and uroa_control
t_end_angle_df = t_end_angle_df.groupby(["uroa_control", "day"])

# remove the keys for day 5
keys = list(t_end_angle_df.groups.keys())
keys.remove((0,5)) if (0,5) in keys else None
keys.remove((1,5)) if (1,5) in keys else None

for key in keys:
    df = t_end_angle_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")


In [ ]:
#create a separate polar plot for each day
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), sharex=True, sharey=True, figsize=(16,10))
plt.suptitle("end trajectory angle, control vs uroa for each day", fontsize=16)
print(keys)
for i, ax in enumerate(axs.flat):
    group = t_end_angle_df.get_group(keys[i])
    angles = group["end_trajectory_angle"].apply(lambda x: x * (np.pi) / 180)  # convert to radians
    # angles = group["angle"]
    bins = np.linspace(0, 2*np.pi, num=16)
    # bins = np.linspace(0, 360, num=20)
    ax.hist(angles, bins=bins, edgecolor="black", linewidth=0.75)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}")

#save png and svg
plt.savefig(end_trajectory_save_path/"end_trajectory_direction.png")
plt.savefig(end_trajectory_save_path/"end_trajectory_direction.svg")

### histogram for tendency to move upwards

In [ ]:
# create a new df with only the y_up data
y_up_df = led_df[["name", "day", "uroa_control", "y_up"]]

#save df
y_up_save_path = save_path/Path("LED_y_up")
os.makedirs(y_up_save_path, exist_ok = True)
y_up_df.to_csv(y_up_save_path/"y_up_df.csv", index = False)

#group by day and uroa_control
y_up_df = y_up_df.groupby(["uroa_control", "day"])

# remove the keys for day 5 if they exist
keys = list(y_up_df.groups.keys())
keys.remove((0, 5)) if (0, 5) in keys else None
keys.remove((1, 5)) if (1, 5) in keys else None

for key in keys:
    df = y_up_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")
    


In [ ]:
fig, axes = plt.subplots(2, 4, sharex=True, sharey = True, figsize=(16,8))
plt.suptitle("tendency to move up, control vs uroa", fontsize=16)
for i, axes in enumerate(axes.flat):
    key = keys[i]
    day_df = y_up_df.get_group(key)
    bins = np.linspace(0, 100, 20)
    # print(day_df.head(5))
    sns.histplot(day_df["y_up"], ax=axes, bins = bins)
    axes.set_title(f"Day {key[1]}")
    axes.set_xlabel("% of frames moving up")
    axes.set_ylabel("number of fish")

#save png and svg
plt.savefig(y_up_save_path/"y_up_hist.png")
plt.savefig(y_up_save_path/"y_up_hist.svg")


In [ ]:
# box plot from the above data by day, total 4 subplots
fig, axes = plt.subplots(1, 4, sharex=True, sharey = True, figsize=(16,8))
plt.suptitle("tendency to move up, control vs uroa", fontsize=16)
for i, axes in enumerate(axes.flat):
    keys_for_day = [keys[i], keys[i+4]]
    day_uro_df = y_up_df.get_group(keys_for_day[1])
    day_control_df = y_up_df.get_group(keys_for_day[0])
    # plot boxplot using sns for both control and urop on same plot
    sns.boxplot(data=[day_control_df["y_up"], day_uro_df["y_up"]], ax=axes)
    axes.set_title(f"Day {keys_for_day[0][1]}")
    #set x axis to control and uroa instead of 0 and 1
    axes.set_xticklabels(["control", "uroa"])
    axes.set_xlabel("control vs uroa")
    axes.set_ylabel("percent of frames moving up")

# save png and svg    
plt.savefig(y_up_save_path/"y_up_day_comaprison_boxplot.png")
plt.savefig(y_up_save_path/"y_up_day_comaprison_boxplot.svg")

In [ ]:
# box plot from the above data, all control in 1 plot, all uro in 1 plot, total 2 subplots
fig, axes = plt.subplots(1, 2, sharex=True, sharey = True, figsize=(16,8))
plt.suptitle("tendency to move up, control vs uroa", fontsize=16)
for i, axes in enumerate(axes.flat):
    keys_for_group = [key for key in keys if key[0] == i]

    #get multiple groups
    day1_df = y_up_df.get_group(keys_for_group[0])
    day2_df = y_up_df.get_group(keys_for_group[1])
    day3_df = y_up_df.get_group(keys_for_group[2])
    day4_df = y_up_df.get_group(keys_for_group[3])

    #plot boxplot
    sns.boxplot(data=[day1_df["y_up"], day2_df["y_up"], day3_df["y_up"], day4_df["y_up"]], ax=axes)
    axes.set_title(f"{'uroa' if i == 1 else 'control'}")
    #set x axis to days instead of 0, 1, 2, 3
    axes.set_xticklabels(["day1", "day2", "day3", "day4"])
    axes.set_xlabel("days")
    axes.set_ylabel("percent of frames moving up")

# save png and svg
plt.savefig(y_up_save_path/"y_up_group_comaprison_boxplot.png")
plt.savefig(y_up_save_path/"y_up_group_comaprison_boxplot.svg")

### polar scatter plot of magnitude of y moved vs trajectory angle

In [ ]:
mag_y_trajectory_df = led_df[["name", "day", "uroa_control", "y_mag", "trajectory_angle"]]

# save df
mag_y_trajectory_save_path = save_path/Path("LED_mag_y_trajectory_angle")
os.makedirs(mag_y_trajectory_save_path, exist_ok = True)
mag_y_trajectory_df.to_csv(mag_y_trajectory_save_path/"mag_y_trajectory_df.csv", index = False)

# group by day and uroa_control
mag_y_trajectory_df = mag_y_trajectory_df.groupby(["uroa_control", "day"])
keys = list(mag_y_trajectory_df.groups.keys())
keys.remove((0, 5)) if (0, 5) in keys else None
keys.remove((1, 5)) if (1, 5) in keys else None
for key in keys:
    df = mag_y_trajectory_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")

In [ ]:
#create a separate polar plot for each day
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), sharex=True, sharey=True, figsize=(24,12))
plt.suptitle("trajectory angle vs y mag, control vs uroa", fontsize=25)
for i, ax in enumerate(axs.flat):
    group = mag_y_trajectory_df.get_group(keys[i])
    angles = group["trajectory_angle"].apply(lambda x: x * (np.pi) / 180)  # convert to radians
    ax.scatter(angles, group["y_mag"], s=25, c='b', alpha=0.5)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=15)

#save png and svg
plt.savefig(mag_y_trajectory_save_path/"mag_y_trajectory_angle_polar.png")
plt.savefig(mag_y_trajectory_save_path/"mag_y_trajectory_angle_polar.svg")

### starting coords vs trajectory polar plot

This puts LED at the center of the polar plot essentially, and we plot how far does each fish start from the LED. If they are starting from a lower height, they are radially outwards (we want this so that those who started closer are shown as closer to LED, plus their trajectory angle does not matter too much)



In [ ]:
start_y_trajectory_df = led_df[["name","day", "uroa_control", "start_y_from_led", "trajectory_angle"]]

# save df
start_y_trajectory_save_path = save_path/Path("LED_start_y_trajectory_angle")
os.makedirs(start_y_trajectory_save_path, exist_ok = True)
start_y_trajectory_df.to_csv(start_y_trajectory_save_path/"start_y_trajectory_df.csv", index = False)

# group by day and uroa_control
start_y_trajectory_df = start_y_trajectory_df.groupby(["uroa_control", "day"])
keys = list(start_y_trajectory_df.groups.keys())
keys.remove((0, 5)) if (0, 5) in keys else None
keys.remove((1, 5)) if (1, 5) in keys else None
for key in keys:
    df = start_y_trajectory_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")

In [ ]:
#create a separate polar plot for each day
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), sharex=True, sharey=True, figsize=(24,12))

# create the fig,axs but limit the plot from 0 to 180 degrees
plt.suptitle("trajectory angle vs y start from LED, control vs uroa", fontsize=25)
for i, ax in enumerate(axs.flat):
    group = start_y_trajectory_df.get_group(keys[i])
    angles = group["trajectory_angle"].apply(lambda x: x * (np.pi) / 180)  # convert to radians
    ax.scatter(angles, group["start_y_from_led"], s=25, c='b', alpha=0.5)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=15)
    # set radial labels -400 to -50 in steps of 100 instead of 50
    ax.set_yticks([-400, -300, -200, -100])
    # set angular limit to 0-180 degrees
    ax.set_xlim(0, np.pi)
    # set angular labels to 0-180 degrees
    ax.set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])


# save png and svg
plt.savefig(start_y_trajectory_save_path/"start_y_trajectory_angle_polar.png")
plt.savefig(start_y_trajectory_save_path/"start_y_trajectory_angle_polar.svg")


### 4-in-1 polar scatter plot
This plot includes start wrt. LED y position and trajectory angle (same as above) but also includes magnitude of movement (red to blue for more movement to less) and % of frames above start position as size of the dot.

In [ ]:
start_y_trajectory_ymag_yup_df = led_df[["name","day", "uroa_control", "start_y_from_led", "trajectory_angle", "y_mag", "y_up"]]
print(start_y_trajectory_ymag_yup_df.head())
# multiply start y by -1 to make it positive using .loc. 
# This makes led at the inside of the circle
start_y_trajectory_ymag_yup_df["start_y_from_led"] = start_y_trajectory_ymag_yup_df["start_y_from_led"].apply(lambda x: x * -1, )
# multiply trajectory angle by -1 to make it positive using .loc. 
# this makes the trajectory angle go clockwise
start_y_trajectory_ymag_yup_df["trajectory_angle"] = start_y_trajectory_ymag_yup_df["trajectory_angle"].apply(lambda x: x * -1)

# print(start_y_trajectory_ymag_yup_df.head())

# save df
start_y_trajectory_ymag_yup_save_path = save_path/Path("start_y_trajectory_ymag_yup")
os.makedirs(start_y_trajectory_ymag_yup_save_path, exist_ok = True)
start_y_trajectory_ymag_yup_df.to_csv(start_y_trajectory_ymag_yup_save_path/"start_y_trajectory_ymag_yup_df.csv", index = False)

# group by day and uroa_control
start_y_trajectory_ymag_yup_df = start_y_trajectory_ymag_yup_df.groupby(["uroa_control", "day"])
keys = list(start_y_trajectory_ymag_yup_df.groups.keys())
keys.remove((0, 5)) if (0, 5) in keys else None
keys.remove((1, 5)) if (1, 5) in keys else None
for key in keys:
    df = start_y_trajectory_ymag_yup_df.get_group(key)
    print(f"{'uroa' if key[0] == 1 else 'control'} day {key[1]}: {len(df)}")

In [ ]:
#create a separate polar plot for each day
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), sharex=True, sharey=True, figsize=(24,10))

# create the fig,axs
plt.suptitle("trajectory angle vs y start from LED, control vs uroa", fontsize=25)
for i, ax in enumerate(axs.flat):
    group = start_y_trajectory_ymag_yup_df.get_group(keys[i])
    angles = group["trajectory_angle"].apply(lambda x: x * (np.pi) / 180)  # convert to radians
    # size = group["y_up"] #normalise the size of the points to 25 - 75 (max size)
    # size = size.apply(lambda x: (x - size.min()) / (size.max() - size.min()) * 50 / 25)
    color = group["y_mag"] #normalise the color of the points to 0 - 1 (max color)
    color = color.apply(lambda x: (x - color.min()) / (color.max() - color.min()))
    color = color.apply(lambda x: (x, 0, 1-x, 1)) # red means y_up is high, blue means y_up is low
    ax.scatter(angles, group["start_y_from_led"], s=50, c=color, alpha=0.5)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=15)
    # set radial labels -400 to -50 in steps of 100 instead of 50
    # ax.set_yticks([-400, -300, -200, -100])
    ax.set_yticks([0, 100, 200, 300, 400])
    # set angular limit to 0-180 degrees
    ax.set_xlim(0, -np.pi)
    # set angular labels to 0-180 degrees
    ax.set_xticks([0, -np.pi/4, -np.pi/2, -3*np.pi/4, -np.pi])
    ax.set_xticklabels(["0", "45", "90", "135", "180"])
    # show color scale used for y_mag. set values to min and max of y_mag   
    #TODO fix colorbar
    sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=plt.Normalize(vmin=group["y_mag"].min(), vmax=group["y_mag"].max()))
    plt.colorbar(sm, ax=ax, shrink=0.5, label="y_mag")
    plt.tight_layout()

# save png and svg
plt.savefig(start_y_trajectory_ymag_yup_save_path/"start_y_trajectory_angle_ymag_yup.png")
plt.savefig(start_y_trajectory_ymag_yup_save_path/"start_y_trajectory_angle_ymag_yup.svg")


In [ ]:
# creating vertical boxplot for each day for start_y_from_led
fig, axs = plt.subplots(2, 4, figsize=(5,10), sharey=True, sharex=True)
plt.suptitle("start_y_from_led, control vs uroa", fontsize=15)
for i, ax in enumerate(axs.flat):
    group = start_y_trajectory_ymag_yup_df.get_group(keys[i])
    sns.boxplot(x="uroa_control", y="start_y_from_led", data=group, ax=ax)
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=10)
    # remove the x axis label and ticks for all columns
    ax.set_xlabel("")
    ax.set_xticks([])
    # remove the y axis label for all columns
    ax.set_ylabel("") if i not in [0,4] else ax.set_ylabel("start_y_from_led", fontsize=10)

# save png and svg
plt.savefig(start_y_trajectory_ymag_yup_save_path/"start_y_boxplot.png")
plt.savefig(start_y_trajectory_ymag_yup_save_path/"start_y_boxplot.svg")



In [ ]:
# # creating horizontal boxplot for each day for trajectory_angle
fig, axs = plt.subplots(2, 4, figsize=(20,4), sharey=True, sharex=True)
plt.suptitle("trajectory_angle, control vs uroa", fontsize=15)
for i, ax in enumerate(axs.flat):
    group = start_y_trajectory_ymag_yup_df.get_group(keys[i])
    # limit boxplot to 180 degrees
    # ax.set_xlim(0, 190)
    # set the x axis ticks to 0, 45, 90, 135, 180
    # ax.set_xticks([0, 45, 90, 135, 180])
    sns.boxplot(y="uroa_control", x="trajectory_angle", data=group, ax=ax, orient="h")
    ax.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=10)
    # remove the x axis label for upper columns
    ax.set_xlabel("") if i not in [4,5,6,7] else ax.set_xlabel("trajectory_angle", fontsize=10)
    # remove the y axis label and ticks for all columns
    ax.set_ylabel("")
    ax.set_yticks([])

# save png and svg
plt.savefig(start_y_trajectory_ymag_yup_save_path/"trajectory_angle_boxplot.png")
plt.savefig(start_y_trajectory_ymag_yup_save_path/"trajectory_angle_boxplot.svg")

## DABEST plots

In [ ]:
import dabest

In [ ]:
# columns to use for the dabest analysis
parameters = led_df.columns[3:]
print(parameters.values)

# save path for the dabest plots
dabest_save_path = Path(f"./data/visualisation/{EXPERIMENT_SET}/dabest_plots/")
os.makedirs(dabest_save_path, exist_ok=True)


# pandas ignore futurewarning and userwarning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)


### Statistical significance plots using DABEST proportion plots

In [ ]:
significance_save_path = dabest_save_path/"significance/"
os.makedirs(significance_save_path, exist_ok=True)
for parameter in parameters:
    print(parameter)
    # box plot from the above data, all control in 1 plot, all uro in 1 plot, total 2 subplots
    led_parameter_df = led_df[["name", "uroa_control", "day", parameter]]
    before_led_parameter_df = before_led_df[["name","uroa_control", "day", parameter]]
    
    # merge led and before led dataframes on name, uroa_control and day
    parameter_df = pd.merge(led_parameter_df, before_led_parameter_df, on=['name', 'uroa_control', 'day'], suffixes=('', '_before_led'), how='inner')
    parameter_df = parameter_df.rename(columns={parameter: "led", f"{parameter}_before_led": "before_led"})
    # print(parameter_df.head())

    # save df
    parameter_df.to_csv(f"{significance_save_path}/{parameter}_led_vs_before_led.csv", index=False)
    
    # group by uroa_control and day and rename columns to led and before_led
    parameter_df = parameter_df.groupby(["uroa_control", "day"])
    
    # get keys for each group
    keys = parameter_df.groups.keys()
    # remove day 5 for uroa and control and return sorted keys
    keys = sorted([key for key in keys if key[1] != 5])
    
    fig, axes = plt.subplots(2, 4, figsize=(25,10), sharey=True)
    plt.suptitle(f"{parameter_dict[parameter]}, led vs before led", fontsize=22)
    #leave more space between subplots
    fig.subplots_adjust(hspace=0.7)
    for i, axes in enumerate(axes.flat):
        # print(i)
        # get the group for each key
        group = parameter_df.get_group(keys[i])
        #drop na
        group = group.dropna()
        # create dabest object
        dabest_obj = dabest.load(group, idx=("before_led", "led"))#, paired= "baseline", id_col="name")
        # save dabest object tests
        tests_df = dabest_obj.mean_diff.statistical_tests
        tests_df.to_csv(significance_save_path/f"{parameter}_{i}_dabest_tests.csv", index=False)
        # plot the dabest object
        dabest_obj.mean_diff.plot(ax=axes)
        # paired plot
        # make the layout of the plot tight
        plt.tight_layout()
        # set the title for the plot
        axes.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=15)
    plt.savefig(significance_save_path/f"{parameter}_dabest.png")
    plt.savefig(significance_save_path/f"{parameter}_dabest.svg")
    plt.show()
    # save png and svg
    plt.close()

### dabest shared control plots

In [ ]:
shared_control_save_path = dabest_save_path/"shared_control/"
os.makedirs(shared_control_save_path, exist_ok=True)
for parameter in parameters:
    print(parameter)
    led_parameter_df = led_df[["name", "uroa_control", "day", parameter]]
    # then shared control is led day 1, with day 2,3,4 being test. this is done for control and uroa
    led_parameter_df = led_parameter_df.groupby(["uroa_control", "day"])
    # get keys for each group
    keys = led_parameter_df.groups.keys()
    # remove day 5 for uroa and control and return sorted keys
    keys = sorted([key for key in keys if key[1] != 5])

    fig, axes = plt.subplots(2, 1, figsize=(10,10))  
    plt.suptitle(f"{parameter_dict[parameter]}, shared control LED day 1 vs other days", fontsize=22)  
    #leave more space between subplots
    for i, axes in enumerate(axes.flat):
        # differentiate bw control and uro
        keys_to_use = [key for key in keys if key[0] == i]
        # create df with name, and parameter from each group
        df = led_parameter_df.get_group(keys_to_use[0])[["name", parameter]]
        df = df.rename(columns={parameter: f"day_{keys_to_use[0][1]}"})
        for key in keys_to_use[1:]:
            group = led_parameter_df.get_group(key)

            group = group.rename(columns={parameter: f"day_{key[1]}"})
            group = group[["name", f"day_{key[1]}"]]

            # currently only merges on common names, but can be merged on all names for a particular grp
            df = pd.merge(df, group, on="name", how="outer")
        # print(np.sort(df['name'].unique()))
        #save df
        df.to_csv(f"{shared_control_save_path}/{parameter}_shared_control_dabest.csv", index=False)
        shared_control = dabest.load(df, idx=("day_1", "day_2", "day_3", "day_4"), id_col="name")
        
        # CHANGE HERE TO GET CLIFF DELTA OR MEDIAN DIFF
        tests_df = shared_control.median_diff.statistical_tests
        tests_df.to_csv(shared_control_save_path/f"{parameter}_{i}_shared_control_dabest_tests.csv", index=False)
        shared_control.median_diff.plot(ax=axes)
        # set the title for the plot
        axes.set_title(f"{'uroa' if i == 1 else 'control'}", fontsize=15)

    # save png and svg
    plt.savefig(shared_control_save_path/f"{parameter}_shared_control_dabest.png")
    plt.savefig(shared_control_save_path/f"{parameter}_shared_control_dabest.svg")
    plt.show()
    plt.close()
    
    

### paired plots

In [ ]:
paired_plt_save_path = dabest_save_path/"paired/"
os.makedirs(paired_plt_save_path, exist_ok=True)

# DEFINE NUMBER OF BEFORE LED DFs TO CREATE
n = 10

averaged_before_led_df = create_averaged_before_led_df(n, columns, csv_files, led_csv, random_pre_led = True, debug = False)
print(averaged_before_led_df.head())

# save the before led df
averaged_before_led_df.to_csv(paired_plt_save_path/"averaged_before_led_df.csv", index=False)

In [ ]:
for parameter in parameters:
    print(parameter)
    # box plot from the above data, all control in 1 plot, all uro in 1 plot, total 2 subplots
    led_parameter_df = led_df[["name", "uroa_control", "day", parameter]]
    before_led_parameter_df = averaged_before_led_df[["name","uroa_control", "day", parameter]]
    # merge led and before led dataframes on name, uroa_control and day
    parameter_df = pd.merge(led_parameter_df, before_led_parameter_df, on=['name', 'uroa_control', 'day'], suffixes=('', '_before_led'), how='inner')
    parameter_df = parameter_df.rename(columns={parameter: "led", f"{parameter}_before_led": "before_led"})
    # print(parameter_df.head())

    # save df
    parameter_df.to_csv(f"{paired_plt_save_path}/{parameter}_led_vs_before_led.csv", index=False)
    
    # group by uroa_control and day and rename columns to led and before_led
    parameter_df = parameter_df.groupby(["uroa_control", "day"])
    
    # get keys for each group
    keys = parameter_df.groups.keys()
    # remove day 5 for uroa and control and return sorted keys
    keys = sorted([key for key in keys if key[1] != 5])
    
    fig, axes = plt.subplots(2, 4, figsize=(25,10), sharey=True)
    plt.suptitle(f"{parameter_dict[parameter]}, led vs before led", fontsize=22)
    #leave more space between subplots
    fig.subplots_adjust(hspace=0.7)
    for i, axes in enumerate(axes.flat):
        # get the group for each key
        group = parameter_df.get_group(keys[i])
        #drop na
        group = group.dropna()
        # create dabest object
        dabest_obj = dabest.load(group, idx=("before_led", "led"), paired= "baseline", id_col="name")
        # save dabest object tests
        tests_df = dabest_obj.mean_diff.statistical_tests
        tests_df.to_csv(paired_plt_save_path/f"{parameter}_{i}_dabest_tests.csv", index=False)
        # plot the dabest object
        dabest_obj.mean_diff.plot(ax=axes)
        # paired plot
        # make the layout of the plot tight
        plt.tight_layout()
        # set the title for the plot
        axes.set_title(f"Day {keys[i][1]} {'uroa' if keys[i][0] == 1 else 'control'}", fontsize=15)
    # save png and svg
    plt.savefig(paired_plt_save_path/f"{parameter}_dabest.png")
    plt.savefig(paired_plt_save_path/f"{parameter}_dabest.svg")
    plt.show()
    plt.close()

### scatter plot unpaired led uro vs control

In [ ]:
unpaired_LED_save_path = dabest_save_path/"unpaired_LED/"
os.makedirs(unpaired_LED_save_path, exist_ok=True)

for parameter in parameters:
    print(parameter)
    # box plot from the above data, all control in 1 plot, all uro in 1 plot, total 2 subplots
    led_parameter_df = led_df[["name", "uroa_control", "day", parameter]]
    # group by uroa_control and day
    parameter_df = led_parameter_df.groupby(["uroa_control", "day"])
    
    # get keys for each group
    keys = parameter_df.groups.keys()
    # remove day 5 for uroa and control and return sorted keys
    keys = sorted([key for key in keys if key[1] != 5])
    
    fig, axes = plt.subplots(1, 4, figsize=(25,5), sharey=True)
    plt.suptitle(f"{parameter_dict[parameter]}, led", fontsize=22)
    #leave more space between subplots
    fig.subplots_adjust(hspace=0.7)
    for i, axes in enumerate(axes.flat):
        # get the group for each key
        control_grp = parameter_df.get_group((0,keys[i][1])).rename(columns={parameter: "control"}).reset_index(drop=True)
        uroa_grp = parameter_df.get_group((1,keys[i][1])).rename(columns={parameter: "uroa"}).reset_index(drop=True)
        # print(len(control_grp))
        # print(len(uroa_grp))
        
        #rename columns
        # control_grp = control_grp.rename(columns={parameter: "control"})
        # uroa_grp = uroa_grp.rename(columns={parameter: "uroa"})

        # concat the two groups. we only need the parameter column. rename the columns to control and uroa in concat
        group = pd.concat([control_grp["control"], uroa_grp["uroa"]], axis=1)

        # rename columns
        # group = group.rename(columns={parameter+"_control": "control", parameter+"_uroa": "uroa"})
        # create dabest object
        # print(group.head())
        dabest_obj = dabest.load(group, idx=("control", "uroa"))
        # save dabest object tests
        tests_df = dabest_obj.mean_diff.statistical_tests
        tests_df.to_csv(unpaired_LED_save_path/f"{parameter}_{i}_dabest_tests.csv", index=False)
        # plot the dabest object
        dabest_obj.mean_diff.plot(ax=axes)
        # paired plot
        # make the layout of the plot tight
        plt.tight_layout()
        # set the title for the plot
        axes.set_title(f"Day {keys[i][1]}", fontsize=15)
    # save png and svg
    plt.savefig(unpaired_LED_save_path/f"{parameter}_dabest.png")
    plt.savefig(unpaired_LED_save_path/f"{parameter}_dabest.svg")
    plt.show()
    plt.close()

### Cliffs delta --> forest plot 